# Notebook 04 — Entraînement du classifieur d'intents

**Objectif :** Entraîner un modèle qui prédit l'intention d'un message utilisateur.

**Exemple :**
- 'Comment publier une demande ?' → `publish_demand`
- 'Mon artisan ne répond pas' → `complaint`
- 'Quel est le prix pour une plomberie ?' → `price_question`

**Approche :**
1. Sentence-transformers pour encoder les messages en vecteurs
2. Logistic Regression pour classifier
3. MLflow pour tracker l'entraînement

**Pourquoi sentence-transformers ?**
- Comprend le sens des phrases (pas juste les mots-clés)
- Le modèle `paraphrase-multilingual-MiniLM-L12-v2` supporte le français et l'arabe
- Léger et rapide en inférence

## 1. Générer les données d'entraînement

In [1]:
import os, sys, warnings, joblib
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from chatbot.data.knowledge_base import KNOWLEDGE_BASE, ALL_INTENTS

CHATBOT_DIR = ROOT / 'chatbot'
MODEL_DIR   = CHATBOT_DIR / 'models'
MLFLOW_URI  = ROOT / 'mlflow_store'
MODEL_DIR.mkdir(exist_ok=True)

print(f'Base de connaissances : {len(KNOWLEDGE_BASE)} documents')
print(f'Intents : {ALL_INTENTS}')

c:\Users\Azzao\OneDrive\Bureau\projects\ml\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Base de connaissances : 22 documents
Intents : ['manage_complaint', 'publish_demand', 'verify_profile', 'validate_artisan', 'availability', 'account', 'review', 'payment', 'manage_offers', 'complaint', 'improve_score', 'monitor_ai', 'price_question', 'send_offer']


In [2]:
# Générer des variantes de questions pour enrichir l'entraînement
# En production : remplacer/compléter avec de vraies questions des utilisateurs

AUGMENTED_DATA = []

# On utilise la question et les mots-clés comme exemples d'entraînement
for doc in KNOWLEDGE_BASE:
    # La question principale
    AUGMENTED_DATA.append({
        'text':   doc['question'],
        'intent': doc['intent'],
        'role':   doc['role'],
    })
    # Les mots-clés comme phrases courtes supplémentaires
    for kw in doc['keywords']:
        AUGMENTED_DATA.append({
            'text':   kw,
            'intent': doc['intent'],
            'role':   doc['role'],
        })

# Exemples supplémentaires écrits à la main
EXTRA_EXAMPLES = [
    # publish_demand
    ('Je veux trouver un plombier',           'publish_demand',  'client'),
    ('J\'ai besoin d\'un électricien urgent', 'publish_demand',  'client'),
    ('Cherche artisan pour réparer',           'publish_demand',  'client'),
    ('Comment poster une annonce',             'publish_demand',  'client'),
    # price_question
    ('Ça coûte combien une réparation',        'price_question',  'client'),
    ('Prix moyen plombier Casablanca',         'price_question',  'client'),
    ('Estimation pour peinture appartement',  'price_question',  'client'),
    ('C\'est quoi le tarif normal',           'price_question',  'client'),
    # manage_offers
    ('J\'ai reçu plusieurs offres',           'manage_offers',   'client'),
    ('Laquelle choisir parmi les offres',      'manage_offers',   'client'),
    ('L\'artisan a annulé son offre',         'manage_offers',   'client'),
    # complaint
    ('L\'artisan n\'est pas venu',            'complaint',       'client'),
    ('Travaux mal faits',                     'complaint',       'client'),
    ('Je veux être remboursé',                'complaint',       'client'),
    ('Mauvaise expérience avec artisan',      'complaint',       'client'),
    # send_offer
    ('Comment répondre à une demande',        'send_offer',      'artisan'),
    ('Je veux postuler pour ce travail',      'send_offer',      'artisan'),
    ('Envoyer ma candidature',                'send_offer',      'artisan'),
    # improve_score
    ('Pourquoi mon score est bas',            'improve_score',   'artisan'),
    ('Comment avoir plus de clients',         'improve_score',   'artisan'),
    ('Augmenter ma visibilité',               'improve_score',   'artisan'),
    # verify_profile
    ('Comment obtenir le badge vérifié',      'verify_profile',  'artisan'),
    ('Soumettre mes documents',               'verify_profile',  'artisan'),
    # account
    ('Impossible de me connecter',            'account',         'tous'),
    ('Changer mon email',                     'account',         'tous'),
    ('Problème de connexion',                 'account',         'tous'),
]

for text, intent, role in EXTRA_EXAMPLES:
    AUGMENTED_DATA.append({'text': text, 'intent': intent, 'role': role})

df = pd.DataFrame(AUGMENTED_DATA)
print(f'Dataset total : {len(df)} exemples')
print(df['intent'].value_counts())

Dataset total : 157 exemples
intent
publish_demand      22
manage_offers       19
price_question      17
send_offer          15
improve_score       15
account             14
complaint           10
verify_profile       9
review               7
availability         7
payment              6
monitor_ai           6
validate_artisan     5
manage_complaint     5
Name: count, dtype: int64


## 2. Encoder les textes avec sentence-transformers

In [3]:
from sentence_transformers import SentenceTransformer

# Modèle multilingue — comprend le français, l'arabe, et l'anglais
# Téléchargé une seule fois, mis en cache automatiquement
MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
print(f'Chargement du modèle {MODEL_NAME}...')
encoder = SentenceTransformer(MODEL_NAME)

print('Encodage des textes...')
X = encoder.encode(df['text'].tolist(), show_progress_bar=True)
y = df['intent'].values

print(f'\nShape des embeddings : {X.shape}')
print(f'  → {X.shape[0]} exemples × {X.shape[1]} dimensions')

Chargement du modèle paraphrase-multilingual-MiniLM-L12-v2...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5085.71it/s]


Encodage des textes...


Batches: 100%|██████████| 5/5 [00:00<00:00, 10.04it/s]


Shape des embeddings : (157, 384)
  → 157 exemples × 384 dimensions


## 3. Entraîner le classifieur avec MLflow

In [4]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder

# Encoder les labels
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

mlflow.set_tracking_uri(str(MLFLOW_URI))
mlflow.set_experiment('atlasfix-chatbot')

import os
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'

with mlflow.start_run(run_name='intent-classifier-v1') as run:
    RUN_ID = run.info.run_id
    print(f'Run ID : {RUN_ID}')

    params = {'C': 1.0, 'max_iter': 1000, 'solver': 'lbfgs', 'multi_class': 'multinomial'}
    mlflow.log_params(params)
    mlflow.log_param('encoder_model', MODEL_NAME)
    mlflow.log_param('n_train', len(X_train))
    mlflow.log_param('n_intents', len(le.classes_))

    # Entraîner
    clf = LogisticRegression(**params)
    clf.fit(X_train, y_train)

    # Évaluer
    y_pred = clf.predict(X_test)
    f1_macro  = round(f1_score(y_test, y_pred, average='macro'),  4)
    f1_weighted = round(f1_score(y_test, y_pred, average='weighted'), 4)
    accuracy  = round(clf.score(X_test, y_test), 4)

    mlflow.log_metric('f1_macro',    f1_macro)
    mlflow.log_metric('f1_weighted', f1_weighted)
    mlflow.log_metric('accuracy',    accuracy)

    print(f'\n=== MÉTRIQUES ===')
    print(f'  Accuracy    : {accuracy:.4f}')
    print(f'  F1 macro    : {f1_macro:.4f}')
    print(f'  F1 weighted : {f1_weighted:.4f}')
    print()
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    # Sauvegarder le bundle complet
    bundle = {
        'classifier':   clf,
        'encoder':      encoder,
        'label_encoder': le,
        'classes':      list(le.classes_),
        'encoder_model': MODEL_NAME,
        'metrics': {
            'accuracy':    accuracy,
            'f1_macro':    f1_macro,
            'f1_weighted': f1_weighted,
        },
        'run_id': RUN_ID,
    }
    path = MODEL_DIR / 'intent_classifier.pkl'
    joblib.dump(bundle, path)
    mlflow.log_artifact(str(path))
    print(f'Modèle sauvegardé → {path}')

UnsupportedModelRegistryStoreURIException:  Model registry functionality is unavailable; got unsupported URI 'C:\Users\Azzao\OneDrive\Bureau\projects\ml\mlflow_store' for model registry data storage. Supported URI schemes are: ['', 'file', 'databricks', 'databricks-uc', 'uc', 'http', 'https', 'postgresql', 'mysql', 'sqlite', 'mssql']. See https://www.mlflow.org/docs/latest/tracking.html#storage for how to run an MLflow server against one of the supported backend storage locations.

## 4. Test du classifieur

In [ ]:
loaded = joblib.load(MODEL_DIR / 'intent_classifier.pkl')

def predict_intent(text: str) -> dict:
    embedding = loaded['encoder'].encode([text])
    proba     = loaded['classifier'].predict_proba(embedding)[0]
    idx       = proba.argmax()
    return {
        'intent':     loaded['classes'][idx],
        'confidence': round(float(proba[idx]), 3),
        'top3': [
            (loaded['classes'][i], round(float(proba[i]), 3))
            for i in proba.argsort()[::-1][:3]
        ]
    }

tests = [
    'Comment publier une demande de plomberie ?',
    'Ça coûte combien pour repeindre un appartement ?',
    'L\'artisan n\'est pas venu au rendez-vous',
    'Comment améliorer mon score ?',
    'Je veux soumettre mes documents de vérification',
    'Mot de passe oublié',
    'Comment envoyer une offre sur cette demande ?',
]

print('=== TEST DU CLASSIFIEUR ===')
for text in tests:
    result = predict_intent(text)
    conf_bar = '█' * int(result['confidence'] * 20)
    print(f'\n  Message    : {text}')
    print(f'  Intent     : {result["intent"]}  (conf: {result["confidence"]}) {conf_bar}')
    print(f'  Top 3      : {result["top3"]}')

In [ ]:
print('=== RÉSUMÉ ===')
m = loaded['metrics']
print(f"  Accuracy    : {m['accuracy']}")
print(f"  F1 macro    : {m['f1_macro']}  ← métrique principale pour ton rapport")
print(f"  F1 weighted : {m['f1_weighted']}")
print(f"  Intents reconnus : {loaded['classes']}")
print()
print('PROCHAINE ÉTAPE : notebooks/05_build_vectordb.ipynb')
print('  → Indexer la base de connaissances dans ChromaDB pour le RAG')